In [2]:
import os
import pandas as pd
from tqdm import tqdm
import shutil

# Parámetros
carpeta_origen = "images_training_rev1"
archivo_csv = "training_solutions_rev1.csv"
carpeta_destino = "argmax"
max_copias = None  # Cambia a un número si quieres limitar la cantidad

# 1. Leer CSV y calcular la clase argmax por fila
df = pd.read_csv(archivo_csv)
if "GalaxyID" not in df.columns:
    raise ValueError("El archivo CSV debe tener una columna 'GalaxyID'.")

# Quitar columna GalaxyID para argmax
cols = df.drop(columns=["GalaxyID"])
argmax = cols.idxmax(axis=1)
conteos = argmax.value_counts().sort_values(ascending=False)
top5 = conteos.index[:5].tolist()

print("Top 5 clases argmax:", top5)

# 2. Filtrar filas que pertenecen a esas 5 clases
mask = argmax.isin(top5)
df_top5 = df[mask]
argmax_top5 = argmax[mask]

# 3. Crear carpeta destino si no existe
os.makedirs(carpeta_destino, exist_ok=True)

# 4. Copiar imágenes correspondientes
copiados = 0
for i, (idx, fila) in enumerate(tqdm(df_top5.iterrows(), total=len(df_top5), desc="Copiando imágenes argmax")):
    galaxy_id = str(int(fila["GalaxyID"]))  # <-- CORREGIDO
    clase = argmax_top5.iloc[i]
    nombre_archivo = f"{galaxy_id}.jpg"
    origen = os.path.join(carpeta_origen, nombre_archivo)
    if os.path.exists(origen):
        destino_clase = os.path.join(carpeta_destino, clase)
        os.makedirs(destino_clase, exist_ok=True)
        destino = os.path.join(destino_clase, nombre_archivo)
        shutil.copy2(origen, destino)
        copiados += 1
        if max_copias and copiados >= max_copias:
            break
    else:
        print(f"Archivo no encontrado: {origen}")

print(f"Total de imágenes copiadas: {copiados}")

Top 5 clases argmax: ['Class6.2', 'Class1.2', 'Class1.1', 'Class6.1', 'Class1.3']


Copiando imágenes argmax: 100%|██████████| 61578/61578 [00:04<00:00, 13725.58it/s]

Total de imágenes copiadas: 61578
